# MedNorm-VI S0 — Domain adaptation / continued pretraining (Colab workflow)

**Status: `DESIGN_DRAFT`. No S0 target model is selected, and this notebook cannot
run until one is.**

**Colab-only.** Full training is OFF by default. This notebook is reviewable
infrastructure, not a trained model. See docs/training/colab_execution_policy.md.

## Why there is no target model here (Audit 0056a)

Until Audit 0056a this notebook's §7-§9 and §13-§20 were a concrete
**PhoBERT-W2NER** workflow: it set `MODEL = "vinai/phobert-base-v2"`, wrote to
`mention/phobert_w2ner`, and told the operator to copy the result into
`models/checkpoints/full_v1/mention/phobert_w2ner`. E4 PhoBERT-W2NER is
`RETIRED_FROM_ACTIVE_ARCHITECTURE` (Audits 0043-0048, 0051) and must not be
restored, so that workflow is removed.

It has **not** been retargeted to ViHealthBERT, XLM-R or anything else. Retargeting
now would be a model-selection decision made without evidence, and S0's own purpose
is to adapt whichever backbone the final profile actually uses. Picking one here
would prejudge that.

A target may be selected only after **all** of the following exist:

1. **code-bearing and assertion evaluation data** — the governed corpus currently
   carries zero ICD-10 codes, zero RxCUIs and zero assertion labels, so no candidate
   or assertion metric is computable and no domain-adaptation gain is measurable
   (`evaluation.code_linking.REQUIRED_ANNOTATION_ARTIFACT`);
2. **the final ≤9B profile is selected** — spec §17 budgets the whole stack
   together; which backbone deserves continued pretraining depends on which
   backbones ship;
3. **exact model revisions and licences are approved** — spec §21 and Appendix A
   require a pinned revision and a local-path-only, no-download runtime, and the
   licence must permit redistribution in the private-rebuild image.

Until then the cells below fail closed on an unset target rather than defaulting to
a model nobody chose.

## 1. Colab & runtime detection · 2. GPU/RAM/disk report


In [ ]:
import sys
import platform
import shutil

IN_COLAB = 'google.colab' in sys.modules
print('in_colab', IN_COLAB, 'python', platform.python_version())
total, used, free = shutil.disk_usage('/')
print('disk_free_gb', round(free / 1e9, 1))
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print('gpu', torch.cuda.get_device_name(0), 'vram_gb', round(p.total_memory / 1e9, 1))
except ImportError as exc:
    print('torch not installed yet:', exc)


## 3. Configurable Google Drive roots (edit PROJECT_ROOT only)


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/mednorm-vi'   # <-- EDIT THIS
DATA_ROOT = PROJECT_ROOT + '/data'
MODEL_CACHE = PROJECT_ROOT + '/model_cache'
CHECKPOINT_ROOT = PROJECT_ROOT + '/checkpoints/full_v1'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError as exc:
    print('not in Colab:', exc)


## 4. Pinned dependencies


In [ ]:
PINS = ["torch==2.3.1", "transformers==4.44.2", "datasets==2.21.0", "accelerate==0.33.0"]
# !pip -q install ' '.join(PINS)
print('pins', PINS)


## 5. Git commit / config verification


In [ ]:
EXPECTED_COMMIT = '<fill: reviewed repo HEAD>'
SEED = 20260723
print('expected_commit', EXPECTED_COMMIT, 'seed', SEED)


## 6. Governed-corpus + split hash verification


In [ ]:
import json
import hashlib

CORPUS = DATA_ROOT + '/derived/training_corpora/mednorm_vi_training_v1'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

with open(CORPUS + '/manifests/split_manifest.json') as fh:
    sm = json.load(fh)
for split, expected in sm['sha256'].items():
    assert sha256(CORPUS + '/splits/' + split + '.jsonl') == expected, split
    print(split, 'OK', sm['counts'][split])
print('cross_split_family_leakage', sm['cross_split_family_leakage'])


## 7. Model/tokenizer revision variables · 8. Model download (Colab only) · 9. Offline-after-download

`DESIGN_DRAFT` — **no target model is selected.** `S0_TARGET_MODEL`,
`S0_TARGET_REVISION` and `S0_CHECKPOINT_ROLE` are deliberately empty and the cell
below raises until all three are filled in together. See the notebook header for the
three conditions that must hold before a target may be chosen.

In [ ]:
import os

# DESIGN_DRAFT (Audit 0056a). This cell set MODEL = "vinai/phobert-base-v2" — the
# retired E4 backbone — until Audit 0056a. It is NOT retargeted: choosing a
# replacement here would be a model-selection decision taken without the evaluation
# data, the frozen <=9B profile, or the approved revision/licence that such a
# decision requires. All three must be filled together, deliberately.
S0_TARGET_MODEL = ""       # <-- e.g. an approved HF model id, once selected
S0_TARGET_REVISION = ""    # <-- the exact pinned commit sha, never a branch name
S0_CHECKPOINT_ROLE = ""    # <-- the checkpoint role this run produces

_unset = [
    name for name, value in (
        ("S0_TARGET_MODEL", S0_TARGET_MODEL),
        ("S0_TARGET_REVISION", S0_TARGET_REVISION),
        ("S0_CHECKPOINT_ROLE", S0_CHECKPOINT_ROLE),
    ) if not value
]
if _unset:
    raise SystemExit(
        "S0 is a DESIGN_DRAFT: " + ", ".join(_unset) + " are unset. A target may be "
        "selected only after code-bearing and assertion evaluation data exist, the "
        "final <=9B profile is selected, and exact model revisions and licences are "
        "approved. Do not default to a model nobody chose.")

os.environ['HF_HUB_OFFLINE'] = '0'   # 0 for the one-time download; set 1 afterwards
# from transformers import AutoModel, AutoTokenizer
# tok = AutoTokenizer.from_pretrained(S0_TARGET_MODEL, revision=S0_TARGET_REVISION, cache_dir=MODEL_CACHE)
# net = AutoModel.from_pretrained(S0_TARGET_MODEL, revision=S0_TARGET_REVISION, cache_dir=MODEL_CACHE)
print('model', S0_TARGET_MODEL, 'revision', S0_TARGET_REVISION)  # weights never committed


## 10. Smoke mode · 11. Full mode OFF by default · 12. Explicit confirmation


In [ ]:
CONFIG = {
    'seed': SEED, 'batch_size': 8, 'grad_accum': 2, 'mixed_precision': 'bf16',
    'checkpoint_every': 50, 'keep_last_k': 3, 'early_stopping_patience': 3,
    'deterministic': True, 'max_smoke_batches': 3,
}
SMOKE = True
RUN_FULL_TRAINING = False   # never auto-runs
CONFIRM_FULL = ''           # must equal 'YES' to allow full training
if RUN_FULL_TRAINING and CONFIRM_FULL != 'YES':
    raise SystemExit('Set CONFIRM_FULL="YES" to run full training deliberately.')
print('stage', "S0", 'smoke', SMOKE, 'full', RUN_FULL_TRAINING)


## 13. Resume logic · 14. Checkpoint rotation


In [ ]:
import os

# The output role follows the SELECTED target (cell above), which is unset in this
# DESIGN_DRAFT. It was hard-coded to "mention/phobert_w2ner" — the retired E4 role —
# until Audit 0056a.
OUT = CHECKPOINT_ROOT + '/' + S0_CHECKPOINT_ROLE
os.makedirs(OUT, exist_ok=True)
def latest_checkpoint(path):
    cks = sorted(f for f in os.listdir(path) if f.startswith('ckpt-')) if os.path.isdir(path) else []
    return cks[-1] if cks else None
print('resume_from', latest_checkpoint(OUT), '| keep_last_k', CONFIG['keep_last_k'])


## 15. Metrics & logs · 16. Parameter count · 17. Artifact hashes · 18. Checkpoint manifest


In [ ]:
import json

metrics = {'mode': 'smoke', 'note': 'path-verified; not a trained model'}
param_count = None   # fill from net.num_parameters() when the model is loaded
manifest = {
    'manifest_version': 1, 'role': S0_CHECKPOINT_ROLE,
    'base_model': {'name': S0_TARGET_MODEL, 'revision': S0_TARGET_REVISION,
                   'parameter_count': param_count},
    'training': {'git_commit': EXPECTED_COMMIT, 'seed': SEED, 'mode': 'smoke',
                 'dataset_manifest_hash': '<fill>', 'split_manifest_hash': '<fill>',
                 'config_hash': '<fill>'},
    'metrics': metrics, 'status': 'SMOKE_ONLY',
}
with open(OUT + '/checkpoint_manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('wrote manifest', OUT)


## 19. Drive export · 20. Return-to-repository instructions

`DESIGN_DRAFT` — these instructions apply only once an S0 target has been selected
through the three conditions in the notebook header. They named the retired E4 role
`models/checkpoints/full_v1/mention/phobert_w2ner` until Audit 0056a.

Copy `OUT` into `models/checkpoints/full_v1/<S0_CHECKPOINT_ROLE>` in the repo
(weights git-ignored; manifest reviewable). The role must be one the canonical
expert specification declares — `mention_factory/expert_spec.py` is the single
source of truth for which roles exist, and a role no expert owns will not satisfy
readiness. Then run local validation:
`model_registry.cli --profile full --require-local-paths`, `pytest -q`,
`phase1c_foundation.cli doctor`. Do NOT commit restricted base-model weights.